## **Import & Setup**

In [1]:
from datasets import load_dataset
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorForLanguageModeling, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel, PeftConfig
import torch
import gc
import sys
import json
import os
from tqdm import tqdm
from huggingface_hub import login
import numpy as np
import time

In [2]:
!nvidia-smi

Sat May 30 22:33:41 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 545.29.06              Driver Version: 545.29.06    CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A40                     Off | 00000000:CA:00.0 Off |                    0 |
|  0%   35C    P0              75W / 300W |   4222MiB / 46068MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [3]:
# Log in to HuggingFace:
login("hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

In [4]:
print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python version: 3.10.12 (main, Feb  4 2025, 14:57:36) [GCC 11.4.0]
PyTorch version: 2.7.0+cu126
CUDA available: True
GPU: NVIDIA A40


## **Load Tokenizer & Model**

In [5]:
MODEL_PATH = "../../1. LLM Fine-Tuning/LLM Models/2. 5 Epochs/fine-tuned-mistralai-Mistral-7B-Instruct-v0.2"

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)
#tokenizer.add_special_tokens({"pad_token": "[PAD]"})

In [7]:
# Load PEFT config to find base model
peft_config = PeftConfig.from_pretrained(MODEL_PATH)
base_model_name = peft_config.base_model_name_or_path

In [8]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [9]:
base_model  = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    quantization_config=quant_config
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [10]:
# Resize to include [PAD]
base_model.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(32001, 4096)

In [11]:
# Load adapter on top of resized base model
model = PeftModel.from_pretrained(base_model, MODEL_PATH)

## **Load the Inference Data**

In [12]:
# === Load your evaluation prompts ===
with open("../Files/inference_dataset.jsonl", "r", encoding="utf-8") as f:
    eval_data = [json.loads(line) for line in f]

## **Perfrom Inferefnce**

### old stuff

In [ ]:
batch_size = 1
results = []

prompts = [ex["prompt"] for ex in eval_data]
expected_completions = [ex["completion"] for ex in eval_data]

for i in tqdm(range(0, len(prompts), batch_size)):
    batch_prompts = [f"<s>[INST] {p.strip()} [/INST]" for p in prompts[i:i+batch_size]]
    batch_expected = expected_completions[i:i+batch_size]

    # Tokenize batched inputs
    inputs = tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024,  # optional safety cutoff
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
        )

    generated_texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    # Strip prompt and [INST] parts from output
    for j, full_text in enumerate(generated_texts):
        if '[/INST]' in full_text:
            response = full_text.split('[/INST]')[-1].strip()
        else:
            response = full_text.strip()

        results.append({
            "prompt": prompts[i + j],
            "expected_completion": batch_expected[j],
            "model_output": response
        })

### new stuff

In [13]:
def run_one(p):
    text = f"<s>[INST] {p.strip()} [/INST]"
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    return full.split("[/INST]")[-1].strip() if "[/INST]" in full else full.strip()

In [14]:
# extract prompts and expected completions for evaluation
prompts = [ex["prompt"] for ex in eval_data]
expected = [ex["completion"] for ex in eval_data]

In [15]:
# warm-up: absorbs CUDA kernel-compile cost, NOT timed
_ = run_one(prompts[0])

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [17]:
# run inference for all and measure time per query
results, per_q = [], []
for p, e in zip(tqdm(prompts), expected):
    t0 = time.perf_counter()
    resp = run_one(p)
    per_q.append(time.perf_counter() - t0)
    results.append({"prompt": p, "expected_completion": e, "model_output": resp})

print(f"queries={len(per_q)}  total={sum(per_q):.1f}s  "
      f"mean/q={np.mean(per_q):.2f}s  median/q={np.median(per_q):.2f}s")

100%|███████████████████████████████████████████████████████████████████████████████████████| 159/159 [1:14:33<00:00, 28.13s/it]

queries=159  total=4472.7s  mean/q=28.13s  median/q=28.22s


In [18]:
# save results
OUTPUT_JSONL = "../Files/inference_outputs_llm.jsonl"

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")